In [0]:
# Bronze ingestion parameters

dbutils.widgets.text("file_id", "")
dbutils.widgets.text("adf_run_id", "")
dbutils.widgets.text("batch_id", "")

dbutils.widgets.text("source_container", "bronze")
dbutils.widgets.text("source_folder", "patients")
dbutils.widgets.text("file_name", "patients.csv")
dbutils.widgets.text("storage_account", "sthealthcarede2026")

dbutils.widgets.text("load_type", "INCREMENTAL")

dbutils.widgets.text("target_schema", "bronze")
dbutils.widgets.text("target_table", "patients_raw")


# Read parameters

file_id = dbutils.widgets.get("file_id").strip()
adf_run_id = dbutils.widgets.get("adf_run_id").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

source_container = dbutils.widgets.get("source_container").strip()
source_folder = dbutils.widgets.get("source_folder").strip("/")
source_file = dbutils.widgets.get("file_name").strip()
storage_account = dbutils.widgets.get("storage_account").strip()

load_type = dbutils.widgets.get("load_type").strip().upper()

target_schema = dbutils.widgets.get("target_schema").strip()
target_table = dbutils.widgets.get("target_table").strip()

# Construct paths

# Original source path from file_control metadata
landing_path = (
    f"abfss://landing@{storage_account}"
    f".dfs.core.windows.net/{source_folder}/{source_file}"
)

# File copied by ADF into the Bronze container
bronze_path = (
    f"abfss://bronze@{storage_account}"
    f".dfs.core.windows.net/{source_folder}/{source_file}"
)

full_table_name = (
    f"dbw_healthcare_de_dev.{target_schema}.{target_table}"
)

print("File ID:", file_id)
print("ADF Run ID:", adf_run_id)
print("Batch ID:", batch_id)
print("Landing path:", landing_path)
print("Source path:", bronze_path)
print("Load type:", load_type)
print("Target table:", full_table_name)

In [0]:
# source file partial path

path = f"{source_folder}/{source_file}"

print(path)

# target table path

target_path = f"{target_schema}.{target_table}"
print(target_path)


In [0]:

# Read CSV from Bronze container

print("Original landing path:", landing_path)
print("Bronze input path:", bronze_path)

df_bronze_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_path)
)

print("Total records:", df_bronze_raw.count())
print("Total columns:", len(df_bronze_raw.columns))


In [0]:

# ============================================================
# BRONZE SCHEMA ALIGNMENT
# ============================================================

from pyspark.sql import functions as F

# Define the expected source columns and data types
expected_schema = {
    "patient_id": "string",
    "first_name": "string",
    "last_name": "string",
    "gender": "string",
    "date_of_birth": "date",
    "state": "string"
}

# Check required source columns before transformation
missing_columns = set(expected_schema) - set(df_bronze_raw.columns)

if missing_columns:
    raise ValueError(
        f"Missing required source columns: {sorted(missing_columns)}"
    )

# Reject unexpected columns rather than silently dropping them
unexpected_columns = set(df_bronze_raw.columns) - set(expected_schema)

if unexpected_columns:
    raise ValueError(
        f"Unexpected source columns: {sorted(unexpected_columns)}"
    )

# Convert incoming columns to expected data types
df_typed = df_bronze_raw.select(
    F.col("patient_id").cast("string").alias("patient_id"),
    F.col("first_name").cast("string").alias("first_name"),
    F.col("last_name").cast("string").alias("last_name"),
    F.col("gender").cast("string").alias("gender"),
    F.to_date(
        F.col("date_of_birth").cast("string"),
        "yyyy-MM-dd"
    ).alias("date_of_birth"),
    F.col("state").cast("string").alias("state")
)
print("df_bronze_raw type:", type(df_bronze_raw))
print("df_typed type:", type(df_typed))

# Validate date conversion
invalid_dates = (
    df_bronze_raw
    .filter(F.col("date_of_birth").isNotNull())
    .count()
    -
    df_typed
    .filter(F.col("date_of_birth").isNotNull())
    .count()
)

if invalid_dates > 0:
    raise ValueError(
        f"Invalid date_of_birth values found: {invalid_dates}"
    )

# Add ingestion metadata
df_bronze = (
    df_typed
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("source_file", F.lit(bronze_path))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("file_id", F.lit(file_id))
    .withColumn("adf_run_id", F.lit(adf_run_id))
)

# Explicitly align with the target Delta table column order
target_columns = [
    "patient_id",
    "first_name",
    "last_name",
    "gender",
    "date_of_birth",
    "state",
    "batch_id",
    "source_file",
    "ingestion_timestamp",
    "file_id",
    "adf_run_id"
]

df_bronze = df_bronze.select(*target_columns)

bronze_count = df_bronze.count()

print("Bronze schema alignment completed")
print("Source records:", bronze_count)
print("Bronze columns:", len(df_bronze.columns))


In [0]:
display(df_bronze)

In [0]:
bronze_count =  df_bronze.count() 
print("bronze record count: ", bronze_count)
print("bronze dataframe column count: ", len(df_bronze.columns))

required_columns = {"patient_id", "batch_id", "source_file"}

actual_columns = set(df_bronze.columns)
print("Actual Columns: ", actual_columns)


# missing columns
# Business rule: If missing_columns is not empty, raise a ValueError.
missing_columns = required_columns - actual_columns
print("Missing columns: ", missing_columns)


if missing_columns:
    raise ValueError(f"Missing required columns : {sorted(missing_columns)}")

if bronze_count == 0:
    raise ValueError("No records found in bronze dataframe")


from pyspark.sql.functions import col
df_patientID_null = (
    df_bronze
    .filter(
        F.col("patient_id").isNull()
        | (F.trim(F.col("patient_id")) == "")
    )
)
null_patientids_count = df_patientID_null.count()
print("Null PatientIDs count: ", null_patientids_count)

if null_patientids_count >0:
    raise ValueError("Null patientID's are present in bronze dataframe")

df_duplicate_patients = (
    df_bronze
    .groupBy(col("patient_id"))
    .count()
    .filter(col('count') > 1)
)
duplicate_patients_count = df_duplicate_patients.count()
print("duplicate patients count: ", duplicate_patients_count)

if duplicate_patients_count > 0:
    raise ValueError("duplicate patientID's are present in bronze dataframe")

print("Bronze validation passed")

In [0]:
catalog_name = "dbw_healthcare_de_dev"

full_table_name = f"{catalog_name}.{target_schema}.{target_table}"

print("Target Delta table:", full_table_name)


In [0]:

from pyspark.sql import functions as F
import uuid

# ============================================================
# STEP 1: VALIDATE PARAMETERS
# ============================================================

if load_type not in ("FULL", "INCREMENTAL"):
    raise ValueError(f"Unsupported load type: {load_type}")

if not file_id or not adf_run_id or not batch_id:
    raise ValueError(
        "file_id, adf_run_id, and batch_id are required"
    )

if not spark.catalog.tableExists(full_table_name):
    raise ValueError(
        f"Bronze target table does not exist: {full_table_name}"
    )

# ============================================================
# STEP 2: VALIDATE FILE OWNERSHIP
# ============================================================

ownership = spark.sql("""
    SELECT file_id
    FROM dbw_healthcare_de_dev.etl.file_control
    WHERE file_id = :file_id
      AND adf_run_id = :adf_run_id
      AND batch_id = :batch_id
      AND source_file = :source_file
      AND pipeline_status = 'IN_PROGRESS'
      AND processing_stage = 'BRONZE'
      AND is_enabled = TRUE
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id,
    "batch_id": batch_id,
    "source_file": landing_path
}).collect()

if len(ownership) != 1:
    raise RuntimeError(
        "Bronze ingestion rejected: file is not owned "
        "by the current ADF run or metadata does not match"
    )

print("File ownership validation passed")

# ============================================================
# STEP 3: PREVENT DUPLICATE FILE INGESTION
# ============================================================

existing_file_count = (
    spark.table(full_table_name)
    .filter(
        (F.col("file_id") == file_id)
        | (F.col("source_file") == bronze_path)
    )
    .limit(1)
    .count()
)

if existing_file_count > 0:
    raise RuntimeError(
        f"File already ingested: file_id={file_id}, "
        f"source_file={bronze_path}. "
        "Reconcile existing Bronze records before retrying."
    )

print("Duplicate file validation passed")

# ============================================================
# STEP 4: START BRONZE AUDIT
# ============================================================

bronze_audit_id = str(uuid.uuid4())

spark.sql("""
    CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
        'START',
        :audit_id,
        :file_id,
        :adf_run_id,
        'BRONZE',
        NULL,
        NULL,
        NULL,
        NULL
    )
""", args={
    "audit_id": bronze_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

print("Bronze audit started:", bronze_audit_id)

# ============================================================
# STEP 5: APPEND TO BRONZE DELTA
# ============================================================

try:
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(full_table_name)
    )

except Exception as write_error:

    # The write may have committed even if an error was returned.
    # Inspect the rows currently visible for this file.

    try:
        committed_rows = (
            spark.table(full_table_name)
            .filter(F.col("file_id") == file_id)
            .count()
        )

    except Exception as verification_error:
        raise RuntimeError(
            "Bronze write failed and commit status could not "
            "be verified. Manual reconciliation required."
        ) from verification_error

    if committed_rows > 0:
        raise RuntimeError(
            f"Bronze write returned an error, but "
            f"{committed_rows} rows exist for file_id={file_id}. "
            "Manual reconciliation required."
        ) from write_error

    # No rows are currently visible. Record an attempt failure,
    # but do not assume that automatic retry is safe.

    try:
        spark.sql("""
            CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
                'FAILED',
                :audit_id,
                :file_id,
                :adf_run_id,
                'BRONZE',
                :source_count,
                0,
                0,
                :error_message
            )
        """, args={
            "audit_id": bronze_audit_id,
            "file_id": file_id,
            "adf_run_id": adf_run_id,
            "source_count": bronze_count,
            "error_message": str(write_error)[:1000]
        }).collect()

    except Exception as audit_error:
        raise RuntimeError(
            "Bronze write failed and FAILED audit could not "
            "be recorded. Manual reconciliation required."
        ) from audit_error

    raise RuntimeError(
        f"Bronze ingestion failed for file_id={file_id}. "
        "FAILED audit recorded. Review Delta history "
        "before retrying."
    ) from write_error

print("Bronze Delta append returned successfully")

# ============================================================
# STEP 6: VERIFY COMMITTED RECORD COUNT
# ============================================================

# Deliberately outside the write exception handler:
# a verification failure must not be classified as a write failure.

try:
    committed_count = (
        spark.table(full_table_name)
        .filter(F.col("file_id") == file_id)
        .count()
    )

except Exception as verification_error:
    raise RuntimeError(
        f"Bronze write returned successfully for file_id={file_id}, "
        "but post-write verification failed. "
        "Manual reconciliation required."
    ) from verification_error

if committed_count != bronze_count:
    raise RuntimeError(
        f"Bronze row count mismatch for file_id={file_id}. "
        f"Expected={bronze_count}, Actual={committed_count}. "
        "Manual reconciliation required."
    )

print("Bronze row count verification passed:", committed_count)

# ============================================================
# STEP 7: RECORD SUCCESS AUDIT
# ============================================================

try:
    spark.sql("""
        CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
            'SUCCESS',
            :audit_id,
            :file_id,
            :adf_run_id,
            'BRONZE',
            :source_count,
            :processed_count,
            :rejected_count,
            NULL
        )
    """, args={
        "audit_id": bronze_audit_id,
        "file_id": file_id,
        "adf_run_id": adf_run_id,
        "source_count": bronze_count,
        "processed_count": committed_count,
        "rejected_count": 0
    }).collect()

except Exception as audit_error:
    raise RuntimeError(
        f"Bronze data committed for file_id={file_id}, "
        "but SUCCESS audit failed or its result is uncertain. "
        "Reconcile audit history before retrying ingestion."
    ) from audit_error

# ============================================================
# STEP 8: COMPLETION LOG
# ============================================================

print("Bronze ingestion and SUCCESS audit completed")

# ============================================================
# STEP 9: ADVANCE FILE FROM BRONZE TO SILVER
# ============================================================

spark.sql("""
    CALL dbw_healthcare_de_dev.etl.usp_update_file_stage(
        :file_id,
        :adf_run_id,
        'BRONZE'
    )
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

# Verify the stage transition
stage_rows = spark.sql("""
    SELECT pipeline_status, processing_stage
    FROM dbw_healthcare_de_dev.etl.file_control
    WHERE file_id = :file_id
      AND adf_run_id = :adf_run_id
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

if (
    len(stage_rows) != 1
    or stage_rows[0]["pipeline_status"] != "IN_PROGRESS"
    or stage_rows[0]["processing_stage"] != "SILVER"
):
    raise RuntimeError(
        "Bronze completed, but SILVER stage transition "
        "could not be verified. Manual reconciliation required."
    )

print("File successfully advanced to SILVER")

print("File ID:", file_id)
print("ADF Run ID:", adf_run_id)
print("Batch ID:", batch_id)
print("Source file:", bronze_path)
print("Source records:", bronze_count)
print("Committed records:", committed_count)
print("Audit ID:", bronze_audit_id)
